# 🚗 GREENSM HUMAN-IN-THE-LOOP ACTIVE LEARNING LAB (M50 - V4.0)
## Notebook Hướng dẫn Chạy Pipeline Toàn diện trên Google Colab

Notebook này hướng dẫn bạn thực thi toàn bộ quy trình **Active Learning 2 Tầng (Two-Tier HITL)** cho bài toán nhận diện xe điện **GreenSM (5-Class Taxonomy: `greensm`, `car`, `motorcycle`, `bus`, `truck`)**.

### ⚙️ Thiết lập Phần cứng Ban đầu:
- Vào menu: **Runtime** -> **Change runtime type** -> Chọn **T4 GPU** (hoặc A100/L4 nếu có Colab Pro).
- Kiểm tra GPU trước khi bắt đầu.

In [ ]:
# [CELL 1] Kiểm tra cấu hình GPU
!nvidia-smi

### 📦 Bước 0: Kết nối Google Drive & Clone Codebase & Cài đặt Môi trường

In [ ]:
# [CELL 2] Mount Google Drive để lưu trữ checkpoint và báo cáo lâu dài
from google.colab import drive
drive.mount('/content/drive')

# Cài đặt các thư viện phụ thuộc của V4.0 Pipeline
!pip install -q ultralytics imagehash opencv-python-headless scikit-learn pyyaml requests openvino

In [ ]:
# [CELL 3] Clone Repo hoặc đồng bộ mã nguồn
import os
%cd /content

if not os.path.exists('BuildPhase'):
    !git clone https://github.com/Wanlee199/BuildPhase.git

%cd /content/BuildPhase
!ls -la

### 📂 Chuẩn bị Dữ liệu (Dùng `vovong.zip` có sẵn hoặc thư mục ảnh từ Google Drive)
Nếu bạn đã có `vovong.zip` trong repo hoặc Drive, giải nén vào thư mục `images/`.

In [ ]:
# [CELL 4] Chuẩn bị kho ảnh đầu vào (Unpack Data)
import zipfile, os, shutil

os.makedirs('images', exist_ok=True)

# Nếu có vovong.zip trong repo:
if os.path.exists('vovong.zip'):
    print("📦 Đang giải nén vovong.zip vào thư mục images/...")
    with zipfile.ZipFile('vovong.zip', 'r') as zip_ref:
        zip_ref.extractall('data_extracted')
    
    # Copy ảnh từ data_extracted/images/train vào images/
    src_train = 'data_extracted/images/train'
    if os.path.exists(src_train):
        for fname in os.listdir(src_train):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                shutil.copy2(os.path.join(src_train, fname), os.path.join('images', fname))
    print(f"✅ Đã chuẩn bị xong {len(os.listdir('images'))} ảnh trong thư mục images/")
else:
    print("ℹ️ Vui lòng tải dữ liệu ảnh của bạn vào thư mục 'images/' hoặc trỏ --data-dir đến Google Drive!")

---
## 🚀 PHƯƠNG ÁN A: Chạy Tự Động Toàn Bộ 6 Bước (1-Click Master Runner)

In [ ]:
# [CELL 5] Chạy tự động trọn gói Master Runner V4.0
!python src/colab_runner.py \
    --data-dir images \
    --output-dir outputs \
    --top-k 200 \
    --epochs 30

---
## 🔍 PHƯƠNG ÁN B: Chạy Từng Bước Độc Lập (Step-by-Step Deep-Dive)

### 🟢 BƯỚC 1: Dual-Stream FastDedup & Metadata Profiling 5 Lát cắt
- Lọc frame trùng lặp bằng pHash (Hamming distance < 5)
- Ưu tiên giữ frame có nhiều xe (Rule Keep-Most-Vehicles)
- Phân loại 5 lát cắt khó: Ban đêm (Night), Mờ nhòe (Blur), Xe nhỏ (Small Object), Che khuất (Occlusion), Mật độ cao (Density).

In [ ]:
# [CELL 6] Bước 1: Data Profiling
!python src/data_profiling.py \
    --data-dir images \
    --output-json outputs/dataset_metadata.json \
    --hash-thresh 5

### 🟢 BƯỚC 2: Active Mining 5 Classes & Lọc Đa Dạng Core-Set
- Chấm điểm Target-Class Confusion Margin giữa GreenSM và Car thường: $U_i = 1 - |P(greensm) - P(car)|$
- Áp dụng giảm chiều PCA (512D -> 32D) và giải thuật **k-Center Greedy (Core-Set)**
- Phân luồng: Tier 1 Auto-Accept (Confidence >= 0.85) vs Tier 2 CVAT Review.

In [ ]:
# [CELL 7] Bước 2: Active Selection
!python src/active_selection.py \
    --metadata-json outputs/dataset_metadata.json \
    --top-k 200 \
    --output-json outputs/active_selection_results.json

### 🟢 BƯỚC 3: Đóng gói Pre-labels CVAT & Assembly Dataset V1 (5 Classes)
- Đóng gói nhãn mồi để annotator sửa nhanh (tiết kiệm ~68% thời gian)
- Tự động tạo Dataset V1 chuẩn format YOLO với 5 classes: `greensm`, `car`, `motorcycle`, `bus`, `truck`
- Phân chia tập `train` và `val` an toàn để phục vụ huấn luyện.

In [ ]:
# [CELL 8] Bước 3: Assembly Dataset
!python src/cvat_automation.py \
    --action assemble \
    --selection-json outputs/active_selection_results.json \
    --output-dir outputs/dataset_v1

### 🟢 BƯỚC 4: Huấn luyện Tích lũy YOLO11n Multi-Seed (Seeds: 42, 123, 456)
- Cố định backbone 10 layers đầu (Backbone Freezing)
- Tối ưu hóa bằng AdamW
- Chạy multi-seed để tính độ lệch chuẩn tự nhiên $\sigma_{seed}$
- Tự động tạo file niêm phong `manifest.yaml` đảm bảo khả năng tái lập 100%.

In [ ]:
# [CELL 9] Bước 4: Train Multi-Seed YOLO11n
!python src/train_yolo11n.py \
    --data-yaml outputs/dataset_v1/dataset.yaml \
    --seeds 42 123 456 \
    --epochs 30 \
    --freeze-layers 10 \
    --project-dir outputs/runs \
    --exp-id greensm_v1

### 🟢 BƯỚC 5: Đánh giá Đa Lát Cắt & Xuất Báo Cáo Khung 3 Trụ Cột ROI
- Đánh giá mô hình trên 4 nhánh A/B Test (Random Baseline vs Active HITL)
- Kiểm định ý nghĩa thống kê: $\Delta mAP > 2\sigma_{seed}$ ($p < 0.05$)
- Xuất báo cáo Markdown hoàn chỉnh `BAO_CAO_BENCHMARK_A_B_TEST.md`.

In [ ]:
# [CELL 10] Bước 5: Đánh giá Benchmark và xuất Báo cáo
!python src/evaluate_benchmark.py \
    --test-yaml outputs/dataset_v1/dataset.yaml \
    --seed-summary-json outputs/runs/greensm_v1_multi_seed_summary.json \
    --output-report outputs/BAO_CAO_BENCHMARK_A_B_TEST.md

### 🟢 BƯỚC 6: Xuất Mô hình ONNX & Lượng Tử Hóa OpenVINO INT8 Triển khai Edge CPU
- Nén mô hình PyTorch xuống OpenVINO INT8 (dung lượng siêu nhẹ ~1.6 MB)
- Đo Benchmark độ trễ (Latency ms) và tốc độ (FPS) cho camera giao thông.

In [ ]:
# [CELL 11] Bước 6: Edge Model Export & OpenVINO INT8
!python src/export_quantize.py \
    --weights outputs/runs/greensm_v1_seed_42/weights/best.pt \
    --output-dir outputs/exported_models

---
## 📊 Hiển Thị Báo Cáo Nghiệm Thu Trực Tiếp trên Colab

In [ ]:
# [CELL 12] Đọc và hiển thị Báo cáo Markdown 3 Trụ Cột ROI
from IPython.display import display, Markdown

report_path = 'outputs/BAO_CAO_BENCHMARK_A_B_TEST.md'
if os.path.exists(report_path):
    with open(report_path, 'r', encoding='utf-8') as f:
        content = f.read()
    display(Markdown(content))
else:
    print("Chưa tìm thấy báo cáo. Hãy chạy Bước 5 trước!")

### 💾 Sao lưu Kết quả về Google Drive

In [ ]:
# [CELL 13] Sao chép toàn bộ outputs về Google Drive
!cp -r outputs /content/drive/MyDrive/GreenSM_Active_Learning_Outputs/
print("✅ Toàn bộ kết quả, trọng số và báo cáo đã được lưu trữ an toàn trên Google Drive!")